In [3]:
#Calculate spearman for EVCoupling Runs

import pandas as pd
ref = pd.read_csv("/n/groups/marks/projects/viral_families/priority-viruses/data/reference_files/viral_dms_reference.csv")

In [4]:
ref

,DMS ID,Viral Family,Virus,Protein,Author,Title,Year,Assay,Type,In ProteinGym,Sequence
0,LASSA_GP_Carr,Arenaviridae,Lassa,GP,Carr,Deep mutational scanning reveals functional co...,2024,fitness,Eukaryotic virus,No,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
1,PESV_POLG_Tsuboyama,Caliciviridae,Porcine enteric sapovirus,POLG,Tsuboyama,Mega-scale experimental analysis of protein fo...,2023,stability,Eukaryotic virus,Yes,ALRDDEYDEWQDIIRDWRKEMTVQQFLDLKERALSGASDPDSQRYN...
2,SARS2_PLPRO_Wu_abundance,Coronaviridae,SARS-CoV-2,PLPRO,Wu,Mutational profiling of SARS-CoV-2 papain-like...,2024,abundance,Eukaryotic virus,No,MEVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIK...
3,SARS2_PLPRO_Wu_activity,Coronaviridae,SARS-CoV-2,PLPRO,Wu,Mutational profiling of SARS-CoV-2 papain-like...,2024,activity,Eukaryotic virus,No,MEVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIK...
4,SARS2_PRD0038_RBD_Starr,Coronaviridae,Bat coronavirus PRD0038,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MKFFILLSLLPFATAQEGCGILSNKSKPALTQYSSSRRGFYYFDDT...
5,SARS2_MRPO_Flynn,Coronaviridae,SARS-CoV-2,MRPO,Flynn,Comprehensive fitness landscape of SARS-CoV-2 ...,2022,fitness,Eukaryotic virus,Yes,SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTS...
6,RmYN02_RBD_Starr,Coronaviridae,Bat coronavirus RmYN02,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MFILLLIGYTAATTCVTGPTTENKQNVSSLMRGVYYPDDIYRSNVN...
7,RsYN04_RBD_Starr,Coronaviridae,Bat coronavirus RsYN04,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MFILLLLPIVLAQQDSCNHIVQLPNSMVRGVYNSGSKVYYPDDINR...
8,SARS2_XBB15_RBD_Taylor,Coronaviridae,SARS-CoV-2 XBB.1.5,RBD,Taylor,Deep mutational scans of XBB.1.5 and BQ.1.1 re...,2023,expression,Eukaryotic virus,No,MFVFLVLLPLVSSQCVNLITRTQSYTNSFTRGVYYPDKVFRSSVLH...
9,SARS2_BA1_SPIKE_Dadonaite,Coronaviridae,SARS-CoV-2 BA.1,SPIKE,Dadonaite,A pseudovirus system enables deep mutational s...,2023,fitness,Eukaryotic virus,No,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...


In [5]:
def generate_mutants(sequence):
    # Define the standard amino acids
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    
    mutants = []; wt = []; pos = []; mut = []   
    for i, wt_aa in enumerate(sequence):
        for mut_aa in amino_acids:
            if mut_aa != wt_aa:
                # Create the mutant in the format ResiduePositionMutant
                mutants.append(f"{wt_aa}{i+1}{mut_aa}")
                wt.append(wt_aa)
                pos.append(i+1)
                mut.append(mut_aa)
    return(pd.DataFrame({"mutant": mutants, "wt" : wt, "pos": pos, "subs": mut}))

def get_evh_merged_df(row):
    main_path = '/n/groups/marks/projects/viral_families/models/evh/'
    theta = 0.99; seq_cov = 50; col_cov = 50
    vir_prot = row['DMS ID']
    
    evh_merged_df = pd.DataFrame()
    for database in ['uniref100', 'uniref90', 'uniref_bfd_mgnify']:
        for bitscore in ['b.5', 'b.3', 'b.1', 'b.05', 'b.04', 'b.03']:
            try:
                prefix_full = vir_prot + "_seqcov" + str(seq_cov) + "_colcov" + str(col_cov) + "_theta" + str(theta)
                model_path = main_path + 'output/' + database + "/" + prefix_full + "/" 
                
                mut_df = pd.read_csv(model_path + prefix_full + "_b." + bitscore.split(".")[-1] + "/mutate/" + prefix_full + "_" + bitscore + "_single_mutant_matrix.csv") 
                mut_df = mut_df[['mutant', 'pos', 'wt', 'subs', 'column_conservation', 'prediction_epistatic', 'prediction_independent']]
                mut_df = mut_df.rename(columns = {'column_conservation' : 'column_conservation_' + bitscore + '_' + database, 
                                                  'prediction_epistatic' : 'prediction_epistatic_' + bitscore + '_' + database, 

                if evh_merged_df.empty:
                    evh_merged_df = mut_df
                else:
                    evh_merged_df = evh_merged_df.merge(mut_df, on = ['pos', 'wt', 'mutant', 'subs'], how = 'outer')
            except:
                pass
    return(evh_merged_df)

def find_mutated_position (target_seq, mutated_sequence):
    x = [a + str(i+1) + b for i, (a,b) in enumerate(zip(target_seq, mutated_sequence)) if a !=b]
    return(",".join(x))

In [ ]:
#evh single mutation scores

#THE SAME METHOD WAS APPLIED TO ALL OTHER MODELS
output_data = []
for i, row in ref.iterrows():
    print('-----', i, row['DMS ID'], '------')
    # Initiate a df for all single mutations using the target sequence
    fin_df = generate_mutants(row['target_seq'])

    #### Step 1: add EVcouplings data ####
    evh_merged_df = get_evh_merged_df(row)    
    if check_df_len(fin_df, evh_merged_df, ['mutant', 'pos', 'wt', 'subs'], 'evh'):
        fin_df = fin_df.merge(evh_merged_df, on=['mutant', 'pos', 'wt', 'subs'], how='left')
    #### Step 2: add DMS data ####
    name = row['DMS ID']
    dms_df = pd.read_csv("/n/groups/marks/projects/viral_families/priority_viruses/data/viral_dms_substitutions/DMS/processed/" + name +'_dms.csv')

    if check_df_len(fin_df, dms_df, ['mutant'], 'dms'):
        fin_df = fin_df.merge(dms_df, on=['mutant'], how='left')
        #limit spearman analysis to mutations with a DMS score
        fin_df = fin_df[fin_df.DMS_score == fin_df.DMS_score].copy()

    # Replace NaN with the average of each column
    fin_df = fin_df.set_index(['mutant', 'wt', 'pos', 'subs'])
    fin_df = fin_df.apply(lambda x: x.fillna(x.mean()), axis=0)
    #### Step 3: calculate Spearman with DMS ####
    spearman_corr = fin_df.corr(method='spearman')['DMS_score']
    spearman = -1 * spearman_corr[[i for i in spearman_corr.index if i.startswith("evol_indices_")]].values[0]
    #fin_df.to_csv('/n/groups/marks/projects/viral_families/results/EVH_singlemutationscores/' + name + ".csv", index=False)
